# Poisson Count Series

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. The *Stochastic Volatility* example smooths a latent autoregressive process observed through a nonlinear node. Here the observations are counts, and we compare the Gaussian approximations the fallback offers with the exact posterior: `LocalLaplace()`, which projects each product of messages on its own, and three schemes that keep a site per edge and make the beliefs agree, `ExpectationPropagation()`, `LaplacePropagation()` and `NaturalGradientPropagation()`.

A latent log-rate follows an autoregressive process, and each count is Poisson with rate $e^{x_t}$:

$$\begin{aligned}
x_1 &\sim \mathcal{N}(0.5, 1),\\
x_t &\sim \mathcal{N}(\varphi x_{t-1}, \sigma^2),\\
y_t &\sim \mathrm{Poisson}(e^{x_t}).
\end{aligned}$$

Because the state is one-dimensional, the exact posterior is within reach: on a fine grid of $x$, the model is a hidden Markov model, and the forward-backward algorithm gives the exact smoothing marginals and the exact evidence $p(y)$.

In [ ]:
using RxInfer, BayesBase, Distributions, SpecialFunctions, StableRNGs
using Plots, PrettyTables, Printf

## The observation node

The count is a node of its own, defined by its log-density alone. Writing it as `y[t] ~ Poisson(exp(x[t]))` would put a deterministic node `exp` before RxInfer's `Poisson` node; the fallback's message from such a node is a Gaussian on the whole real line, which knows nothing of the rate being positive, and Poisson's rules need a positive rate. A node that takes the log-rate directly has no such problem.

In [ ]:
struct LogPoisson{X} <: DiscreteUnivariateDistribution
    x::X
end

BayesBase.logpdf(d::LogPoisson, k) = k * d.x - exp(d.x) - loggamma(k + 1)
BayesBase.insupport(::LogPoisson, k) = true

@define_factor_node(node = LogPoisson, type = Stochastic, interfaces = [:out, :x])

## Data

In [ ]:
φ, σ², T = 0.9, 0.1, 150
rng = StableRNG(42)

x_true = zeros(T)
x_true[1] = rand(rng, Normal(0.5, 1.0))
for t in 2:T
    x_true[t] = φ * x_true[t - 1] + rand(rng, Normal(0.0, sqrt(σ²)))
end
y = [rand(rng, Poisson(exp(x))) for x in x_true]

bar(y; color = :gray, label = "counts yₜ", xlabel = "t", size = (800, 300))

## Inference

In [ ]:
@model function count_series(y, φ, σ²)
    x[1] ~ NormalMeanVariance(0.5, 1.0)
    y[1] ~ LogPoisson(x[1])
    for t in 2:length(y)
        x[t] ~ NormalMeanVariance(φ * x[t - 1], σ²)
        y[t] ~ LogPoisson(x[t])
    end
end

laplace = infer(
    model = count_series(φ = φ, σ² = σ²), data = (y = y,),
    fallback = LocalLaplace(), free_energy = true, report = true,
)

Under belief propagation the transitions keep their exact rules. Every product of messages on the chain, a marginal or a message leaving $x_t$, is projected onto a Gaussian by the Laplace method, at its mode and with the curvature there. The report shows it:

In [ ]:
@assert isempty(RxInfer.diagnostics(laplace.report)) # hide
@assert RxInfer.node_types(laplace.report.approximations)[(LogPoisson, :message, :x)].fallbacks == T # hide
laplace.report.approximations

## The exact smoother

On a grid of 801 points for $x$, the transition is a matrix and each count a vector of likelihoods. A forward pass gives the filtering distributions and the evidence, and a backward pass the smoothing ones.

In [ ]:
function exact_smoother(y; grid = range(-5, 5, length = 801))
    h = step(grid)
    K = [pdf(Normal(φ * u, sqrt(σ²)), v) * h for u in grid, v in grid]
    likelihood(t) = [exp(logpdf(LogPoisson(u), y[t])) for u in grid]
    α, c = zeros(length(grid), T), zeros(T)
    a = [pdf(Normal(0.5, 1.0), u) * h for u in grid] .* likelihood(1)
    c[1] = sum(a); α[:, 1] = a ./ c[1]
    for t in 2:T
        a = (K' * α[:, t - 1]) .* likelihood(t)
        c[t] = sum(a); α[:, t] = a ./ c[t]
    end
    β = ones(length(grid), T)
    for t in (T - 1):-1:1
        β[:, t] = K * (likelihood(t + 1) .* β[:, t + 1]) ./ c[t + 1]
    end
    P = α .* β
    P ./= sum(P; dims = 1)
    m = [sum(P[:, t] .* grid) for t in 1:T]
    s = [sqrt(sum(P[:, t] .* (grid .- m[t]) .^ 2)) for t in 1:T]
    return (; mean = m, sd = s, logevidence = sum(log, c))
end

exact = exact_smoother(y)

## Results

In [ ]:
m_laplace, s_laplace = mean.(laplace.posteriors[:x]), std.(laplace.posteriors[:x])

plt = plot(exact.mean; ribbon = 2exact.sd, fillalpha = 0.2, color = :gray, label = "exact (±2 sd)", xlabel = "t", ylabel = "log-rate xₜ", size = (900, 450), legend = :outerbottom, legendcolumns = 3)
plot!(plt, m_laplace; color = :blue, label = "LocalLaplace()")
plot!(plt, x_true; color = :black, linewidth = 1, label = "true xₜ")

In [ ]:
rms(a) = sqrt(sum(abs2, a) / length(a))
F_laplace = only(laplace.free_energy)
rows = [
    "LocalLaplace()" rms(m_laplace - exact.mean) maximum(abs, m_laplace - exact.mean) maximum(abs, s_laplace - exact.sd) F_laplace;
    "exact" 0.0 0.0 0.0 -exact.logevidence
]
@assert 0.08 < rms(m_laplace - exact.mean) < 0.15 && maximum(abs, s_laplace - exact.sd) < 0.05 # hide
@assert -exact.logevidence + 0.5 < F_laplace # hide
pretty_table(rows; column_labels = ["", "RMS of mean - exact", "max |mean - exact|", "max |sd - exact|", "free energy"], formatters = [fmt__printf("%.4f", 2:5)], fit_table_in_display_horizontally = false)

`LocalLaplace()`'s means are about 0.1 from the exact smoother's. A count of zero, or a count far from the rate, makes the posterior of $x_t$ skewed, and Laplace places its Gaussian at the mode. Its spreads are close to the exact ones.

Its free energy exceeds $-\log p(y)$ by about 0.7. Under belief propagation, a projected marginal and the joints of the transition nodes around it are computed from different messages, so they do not agree exactly. The free energy of such beliefs is then an estimate of $-\log p(y)$, not a bound, and a lower one is not a better fit. The schemes of the next section make them agree.

## Cavity propagation

Under `fallback = ExpectationPropagation()`, each $x_t$ keeps one Gaussian site per edge, and its beliefs are products of sites: its marginal, and each message it sends, the product of the sites of its other edges, called the edge's cavity. The count's message, a log-density, is projected together with its cavity, the two transitions' messages, by moment matching, and the cavity is divided out again: what is left is the count's site. The marginal and the messages to the transitions are then made of the same sites, so the transitions' joints agree with the marginals. Each site depends on its neighbours' through the chain, and the iterations sweep the sites to a fixed point. `LaplacePropagation()` does the same by the Laplace method, the mode and curvature of each tilted belief, the message times its cavity.

`NaturalGradientPropagation()` keeps the same sites, but moves each one by a step rather than projecting it (Lukashchuk et al., *Information Geometry of Message Passing*). The count's site on $x_t$ is a Gaussian, the family of its cavity, and at each iteration it takes a damped heavy-ball step in natural parameters, $v \leftarrow \beta v + \alpha(\eta^\star - \eta)$ and $\eta \leftarrow \eta + v$, with the step size `alpha = 0.5` and the momentum `beta = 0.2` by default, towards its target $\eta^\star = G(q)^{-1}\,\mathrm{Cov}_q[T(x_t), \ell(x_t)]$: the natural gradient of $\mathbb{E}_q[\ell]$, with $\ell$ the count's log-density, $q$ the marginal of $x_t$, and $T$ and $G$ the Gaussian's sufficient statistics and Fisher information. At its fixed point every site is its target. The chain is then a Gaussian tree whose only non-conjugate factors, the counts, are leaves: the beliefs are the Gaussian over the whole path closest to the posterior in $\mathrm{KL}(q \,\|\, p)$, and the free energy is its negative evidence lower bound, an upper bound on $-\log p(y)$. That fixed point is the one conjugate-computation variational inference reaches.

Natural-gradient steps need more sweeps than projections do: thirty iterations let every site settle to the scheme's tolerance, `tol = 1e-8` relative.

In [ ]:
propagate(fallback) = infer(
    model = count_series(φ = φ, σ² = σ²), data = (y = y,), iterations = 30,
    fallback = fallback, free_energy = true, returnvars = (x = KeepLast(),), report = true,
)

ep = propagate(ExpectationPropagation())
lp = propagate(LaplacePropagation())
ngmp = propagate(NaturalGradientPropagation())

The report shows the sites, one per edge of each $x_t$, and how many were projected over the thirty iterations, here for expectation propagation; under `NaturalGradientPropagation()`, it counts the steps of the count's site instead, and says whether each has settled at its target:

In [ ]:
@assert isempty(RxInfer.diagnostics(ep.report)) && isempty(RxInfer.diagnostics(lp.report)) && isempty(RxInfer.diagnostics(ngmp.report)) # hide
@assert all(record -> record.sites.projections > 0, (ep.report.approximations.variables[(:x, t)] for t in 1:T)) # hide
@assert all(record -> record.sites.outcome.method === :natural_gradient, (ngmp.report.approximations.variables[(:x, t)] for t in 1:T)) # hide
ep.report.approximations

In [ ]:
m_ep, s_ep = mean.(ep.posteriors[:x]), std.(ep.posteriors[:x])
m_lp, s_lp = mean.(lp.posteriors[:x]), std.(lp.posteriors[:x])
m_ngmp, s_ngmp = mean.(ngmp.posteriors[:x]), std.(ngmp.posteriors[:x])
F_ep, F_lp, F_ngmp = last(ep.free_energy), last(lp.free_energy), last(ngmp.free_energy)
rows = [
    "LocalLaplace()" rms(m_laplace - exact.mean) maximum(abs, m_laplace - exact.mean) maximum(abs, s_laplace - exact.sd) F_laplace;
    "LaplacePropagation()" rms(m_lp - exact.mean) maximum(abs, m_lp - exact.mean) maximum(abs, s_lp - exact.sd) F_lp;
    "ExpectationPropagation()" rms(m_ep - exact.mean) maximum(abs, m_ep - exact.mean) maximum(abs, s_ep - exact.sd) F_ep;
    "NaturalGradientPropagation()" rms(m_ngmp - exact.mean) maximum(abs, m_ngmp - exact.mean) maximum(abs, s_ngmp - exact.sd) F_ngmp;
    "exact" 0.0 0.0 0.0 -exact.logevidence
]
@assert rms(m_ep - exact.mean) < 1e-4 && maximum(abs, m_ep - exact.mean) < 2e-4 && maximum(abs, s_ep - exact.sd) < 0.005 # hide
@assert rms(m_ngmp - exact.mean) < 5e-4 && maximum(abs, m_ngmp - exact.mean) < 1e-3 && maximum(abs, s_ngmp - exact.sd) < 0.01 # hide
@assert rms(m_ep - exact.mean) < rms(m_ngmp - exact.mean) < rms(m_lp - exact.mean) < rms(m_laplace - exact.mean) && maximum(abs, s_lp - exact.sd) < 0.005 # hide
@assert abs(F_ep + exact.logevidence) < 0.06 && abs(F_lp + exact.logevidence) < 0.06 # hide
@assert 0.05 < F_ngmp + exact.logevidence < 0.15 # hide
pretty_table(rows; column_labels = ["", "RMS of mean - exact", "max |mean - exact|", "max |sd - exact|", "free energy"], formatters = [fmt__printf("%.4f", 2:5)], fit_table_in_display_horizontally = false)

Expectation propagation's means are the exact smoother's to about $10^{-4}$, and its spreads to within a few thousandths. Natural-gradient propagation's means are nearly as close, to a few $10^{-4}$, and its spreads a little smaller than the exact ones, as a variational Gaussian's are: $\mathrm{KL}(q \,\|\, p)$ penalises mass where the posterior has little. Laplace propagation reaches the global Laplace approximation, the mode of the whole posterior and its curvature: its spreads are as close as EP's, and its means are the modes of skewed marginals, closer to the exact means than `LocalLaplace()`'s, but about 0.07 away.

Every belief is now made of the same sites, so the transitions' joints agree with the marginals. The free energies of the three mean two different things. Expectation and Laplace propagation's is expectation propagation's: each node without rules takes its term from its cavities, the other terms are Bethe's, and at the fixed point it is an estimate of $-\log p(y)$, here within 0.06, but not a bound. Natural-gradient propagation's is Bethe's with the counts' average energies under the marginals: the negative evidence lower bound, about 0.1 above $-\log p(y)$, which it cannot cross; the gap is the divergence of the Gaussian path from the posterior. Each is computed once per iteration and settles within a few:

In [ ]:
plot(ep.free_energy; marker = :circle, color = :red, label = "ExpectationPropagation()", xlabel = "iteration", ylabel = "free energy", size = (800, 350))
plot!(lp.free_energy; marker = :circle, color = :blue, label = "LaplacePropagation()")
plot!(ngmp.free_energy; marker = :circle, color = :green, label = "NaturalGradientPropagation()")
hline!([-exact.logevidence]; color = :black, linestyle = :dot, label = "-log p(y), exact")

## Learning the transition precision

Now the precision of the transition, $\tau = 1/\sigma^2$, is unknown, with a $\mathrm{Gamma}(2, 0.2)$ prior, and we factorise $q(x, \tau) = q(x)\,q(\tau)$. The chain keeps its belief propagation, and $\tau$'s update is conjugate: its posterior is a Gamma distribution, which passes the fallback's projection unchanged. We compare `LocalLaplace()` with `NaturalGradientPropagation()`: under mean field, the transitions send variational messages to the $x_t$, Gaussians that are their own sites, and the count's site on each $x_t$ steps as before. Coordinate ascent between the path and its precision is slow, so both run for 200 iterations.

In [ ]:
@model function count_series_τ(y, φ)
    τ ~ Gamma(shape = 2.0, rate = 0.2)
    x[1] ~ NormalMeanVariance(0.5, 1.0)
    y[1] ~ LogPoisson(x[1])
    for t in 2:length(y)
        x[t] ~ NormalMeanPrecision(φ * x[t - 1], τ)
        y[t] ~ LogPoisson(x[t])
    end
end

separate_τ = @constraints begin
    q(x, τ) = q(x)q(τ)
end

initial_τ = @initialization begin
    q(τ) = GammaShapeRate(2.0, 0.2)
end

fit_τ(fallback) = infer(
    model = count_series_τ(φ = φ), data = (y = y,),
    constraints = separate_τ, initialization = initial_τ,
    iterations = 200, fallback = fallback, free_energy = true, report = true,
)

laplace_τ, ngmp_τ = fit_τ(LocalLaplace()), fit_τ(NaturalGradientPropagation())

The report confirms that $\tau$'s posterior is exact and every $x_t$'s is approximated:

In [ ]:
records_τ = ngmp_τ.report.approximations
@assert !RxInfer.is_approximated(records_τ.variables[(:τ, nothing)]) # hide
@assert count(RxInfer.is_approximated, values(records_τ.variables)) == T # hide
@assert isempty(RxInfer.diagnostics(ngmp_τ.report)) && isempty(RxInfer.diagnostics(laplace_τ.report)) # hide
records_τ

For the exact posterior of $\tau$, the forward pass gives the evidence $p(y \mid \tau)$ on a grid of $\tau$ values, a coarser grid of $x$ being enough for it, and its integral over the prior the evidence $p(y)$.

In [ ]:
function logevidence_given(τ; grid = range(-4, 4, length = 321))
    h = step(grid)
    K = [pdf(Normal(φ * u, 1 / sqrt(τ)), v) * h for u in grid, v in grid]
    a = [pdf(Normal(0.5, 1.0), u) * h for u in grid] .* [exp(logpdf(LogPoisson(u), y[1])) for u in grid]
    s = log(sum(a)); a ./= sum(a)
    for t in 2:T
        a = (K' * a) .* [exp(logpdf(LogPoisson(u), y[t])) for u in grid]
        z = sum(a); s += log(z); a ./= z
    end
    return s
end

τs = range(0.5, 60, length = 240)
logposterior_τ = [logpdf(Gamma(2.0, 1 / 0.2), τ) + logevidence_given(τ) for τ in τs]
logevidence_τ = maximum(logposterior_τ) + log(sum(exp.(logposterior_τ .- maximum(logposterior_τ))) * step(τs))
w = exp.(logposterior_τ .- maximum(logposterior_τ)); w ./= sum(w)
Eτ = sum(w .* τs); sdτ = sqrt(sum(w .* (τs .- Eτ) .^ 2))

qτ_laplace, qτ_ngmp = laplace_τ.posteriors[:τ][end], ngmp_τ.posteriors[:τ][end]
F_laplace_τ, F_ngmp_τ = last(laplace_τ.free_energy), last(ngmp_τ.free_energy)
@assert Eτ > 1 / σ² && max(std(qτ_laplace), std(qτ_ngmp)) < sdτ / 3 # hide
@assert abs(mean(qτ_laplace) - Eτ) < 0.5 * sdτ && abs(mean(qτ_ngmp) - Eτ) < 0.5 * sdτ # hide
@assert abs(mean(qτ_ngmp) - Eτ) < abs(mean(qτ_laplace) - Eτ) # hide
@assert -logevidence_τ + 1 < F_ngmp_τ < F_laplace_τ # hide
pretty_table(
    ["exact" Eτ sdτ -logevidence_τ; "LocalLaplace()" mean(qτ_laplace) std(qτ_laplace) F_laplace_τ; "NaturalGradientPropagation()" mean(qτ_ngmp) std(qτ_ngmp) F_ngmp_τ];
    column_labels = ["", "E[τ]", "sd(τ)", "free energy"], formatters = [fmt__printf("%.2f", 2:4)], fit_table_in_display_horizontally = false,
)

The data, 150 counts of about one or two, say little about $\tau$: its exact posterior is wide, and its mean is above the true value 10. Both schemes find a mean within half a standard deviation of the exact one, natural-gradient propagation's the closer, and both are overconfident by a factor of three or more. That is the price of the factorisation $q(x)\,q(\tau)$: the path and its precision are learned as if independent, while a rougher path and a lower precision explain the same counts. It shows in the free energies too: natural-gradient propagation's is the lower, but more than a nat above $-\log p(y)$, the first row. The plot shows the exact posterior and the two Gamma approximations.

In [ ]:
τplot = range(0.5, 45, length = 400)
plot(τs, w ./ step(τs); color = :black, linewidth = 2, label = "exact", xlabel = "τ", ylabel = "density", xlims = (0, 45))
plot!(τplot, pdf.(qτ_laplace, τplot); color = :blue, label = "LocalLaplace()")
plot!(τplot, pdf.(qτ_ngmp, τplot); color = :red, linestyle = :dash, label = "NaturalGradientPropagation()")
vline!([1 / σ²]; color = :gray, linestyle = :dot, label = "true τ")